# Maximum Likelihood Estimation for DFMs

**Goal:** Implement ML estimation using the Kalman filter.

**Time:** 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Simulate data from local level model
np.random.seed(42)
T = 200
true_Q = 0.5
true_H = 1.0

states = np.cumsum(np.random.randn(T) * np.sqrt(true_Q))
y = states + np.random.randn(T) * np.sqrt(true_H)

plt.figure(figsize=(12, 5))
plt.plot(states, 'k-', linewidth=2, label='True State', alpha=0.7)
plt.plot(y, 'o', alpha=0.4, markersize=4, label='Observations')
plt.legend()
plt.title('Simulated Data')
plt.grid(True, alpha=0.3)
plt.show()

print(f"True parameters: Q={true_Q:.2f}, H={true_H:.2f}")

## ML Estimation via Kalman Filter

In [ ]:
def neg_loglik(params, y):
    """Negative log-likelihood for local level model."""
    Q, H = np.exp(params)  # Ensure positive
    
    loglik = 0.0
    a, P = 0.0, 10.0
    
    for t in range(len(y)):
        # Predict
        a_pred = a
        P_pred = P + Q
        
        # Update
        v = y[t] - a_pred
        F = P_pred + H
        K = P_pred / F
        
        a = a_pred + K * v
        P = P_pred - K * F * K
        
        # Add to log-likelihood
        loglik += -0.5 * (np.log(2*np.pi) + np.log(F) + v**2/F)
    
    return -loglik

# Optimize
initial_params = [np.log(0.5), np.log(1.0)]
result = minimize(neg_loglik, initial_params, args=(y,), method='BFGS')

Q_hat, H_hat = np.exp(result.x)

print(f"ML Estimates:")
print(f"  Q: {Q_hat:.3f} (true: {true_Q:.3f})")
print(f"  H: {H_hat:.3f} (true: {true_H:.3f})")
print(f"\nLog-likelihood: {-result.fun:.2f}")
print(f"Optimization successful: {result.success}")

## Visualize Likelihood Surface

In [ ]:
# Create grid
Q_grid = np.linspace(0.1, 1.5, 30)
H_grid = np.linspace(0.3, 2.0, 30)
Q_mesh, H_mesh = np.meshgrid(Q_grid, H_grid)

# Compute likelihood on grid
loglik_grid = np.zeros_like(Q_mesh)
for i in range(len(Q_grid)):
    for j in range(len(H_grid)):
        loglik_grid[j, i] = -neg_loglik([np.log(Q_grid[i]), np.log(H_grid[j])], y)

# Plot
fig = plt.figure(figsize=(12, 5))

# Contour plot
ax1 = fig.add_subplot(121)
contour = ax1.contour(Q_mesh, H_mesh, loglik_grid, levels=20)
ax1.plot(true_Q, true_H, 'r*', markersize=20, label='True')
ax1.plot(Q_hat, H_hat, 'bx', markersize=15, label='ML Estimate', markeredgewidth=3)
ax1.set_xlabel('Q')
ax1.set_ylabel('H')
ax1.set_title('Log-Likelihood Surface')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 3D surface
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(Q_mesh, H_mesh, loglik_grid, alpha=0.7, cmap='viridis')
ax2.set_xlabel('Q')
ax2.set_ylabel('H')
ax2.set_zlabel('Log-Likelihood')
ax2.set_title('Likelihood Surface (3D)')

plt.tight_layout()
plt.show()

print("ML estimation finds the peak of the likelihood surface!")